In [10]:
!pip install ultralytics


In [11]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="urXfMn8nBl4OyNtc054s")
project = rf.workspace("chandus-workspace-z5lky").project("pothole-detection-pz8l4")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


In [12]:
!python train.py --data Pothole-Detection-1/data.yaml --epochs 50 --model yolov8n.pt

Ultralytics 8.4.124 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Pothole-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=civic_twin_defect_model-2, nbs=

In [13]:
!python detect.py --image /content/Pothole-Detection-1/train/images/pothole_1000_jpg.rf.8cb32e542650af90ff8c5c04be4bc87b.jpg --weights runs/detect/civic_twin_defect_model/weights/best.pt

{
  "image": "/content/Pothole-Detection-1/train/images/pothole_1000_jpg.rf.8cb32e542650af90ff8c5c04be4bc87b.jpg",
  "image_width": 640,
  "image_height": 640,
  "defects_found": 1,
  "detections": [
    {
      "defect_type": "Pothole Detection - v5 2026-02-18 9-14pm",
      "confidence": 0.546,
      "bounding_box": {
        "x1": 83.4,
        "y1": 9.8,
        "x2": 640.0,
        "y2": 640.0
      },
      "severity": "Critical",
      "severity_score": 85.9
    }
  ],
  "overall_severity": "Critical"
}


In [14]:
!pkill -f uvicorn

In [ ]:
import subprocess
log_file = open('server_log.txt', 'w')
subprocess.Popen(['uvicorn', 'api:app', '--port', '8001'], stdout=log_file, stderr=subprocess.STDOUT)

<Popen: returncode: None args: ['uvicorn', 'api:app', '--port', '8001']>

In [15]:
import subprocess; subprocess.Popen(['uvicorn', 'api:app', '--port', '8001'])

<Popen: returncode: None args: ['uvicorn', 'api:app', '--port', '8001']>

In [16]:
import time
time.sleep(5)
print("done waiting")

done waiting


In [17]:
import requests
f = open('/content/Pothole-Detection-1/train/images/pothole_1000_jpg.rf.8cce5c4b86cc49b7993203cac691f0e9.jpg', 'rb')
response = requests.post('http://127.0.0.1:8001/detect', files={'file': ('test.jpg', f, 'image/jpeg')})
f.close()
print(response.status_code)
print(response.text)

200
{"image":"temp_uploads/881a1390f8f34c54acf97eab98ccaa19_test.jpg","image_width":640,"image_height":640,"defects_found":1,"detections":[{"defect_type":"Pothole Detection - v5 2026-02-18 9-14pm","confidence":0.731,"bounding_box":{"x1":89.1,"y1":3.8,"x2":640.0,"y2":630.9},"severity":"Critical","severity_score":89.6}],"overall_severity":"Critical"}


In [18]:
import os; os.environ['MODEL_WEIGHTS'] = 'runs/detect/civic_twin_defect_model/weights/best.pt'